# Recommender System Evaluation Metrics

## Learning Objectives

1. **Implement** RMSE and MAE for rating prediction
2. **Compute** precision@k, recall@k, and F1@k for ranking evaluation
3. **Derive** NDCG for graded relevance
4. **Discuss** coverage, novelty, and serendipity beyond accuracy

## Rating Prediction Metrics

When we have explicit ratings, we can measure prediction accuracy:

**RMSE (Root Mean Squared Error):**
$$\text{RMSE} = \sqrt{\frac{1}{|\Omega_{\text{test}}|}\sum_{(u,i)\in\Omega_{\text{test}}}(r_{ui} - \hat{r}_{ui})^2}$$

**MAE (Mean Absolute Error):**
$$\text{MAE} = \frac{1}{|\Omega_{\text{test}}|}\sum_{(u,i)\in\Omega_{\text{test}}}|r_{ui} - \hat{r}_{ui}|$$

RMSE penalizes large errors more than MAE (due to squaring).

In [1]:
import math

def rmse(actual, predicted):
    return math.sqrt(sum((a-p)**2 for a,p in zip(actual,predicted)) / len(actual))

def mae(actual, predicted):
    return sum(abs(a-p) for a,p in zip(actual,predicted)) / len(actual)

# Simulated rating predictions
actual    = [5, 4, 3, 2, 4, 5, 1, 3]
predicted = [4.5, 3.8, 3.1, 2.5, 3.6, 4.8, 1.5, 3.2]
print(f"RMSE: {rmse(actual, predicted):.4f}")
print(f"MAE:  {mae(actual, predicted):.4f}")

# Baseline comparisons
global_mean = sum(actual)/len(actual)
pred_mean = [global_mean]*len(actual)
print(f"""
Baseline (global mean={global_mean:.2f}):""")
print(f"  RMSE: {rmse(actual, pred_mean):.4f}")
print(f"  MAE:  {mae(actual, pred_mean):.4f}")

RMSE: 0.3606
MAE:  0.3250

Baseline (global mean=3.38):
  RMSE: 1.3170
  MAE:  1.1250


In [2]:
def precision_at_k(recommended, relevant, k):
    """Fraction of top-k recommendations that are relevant."""
    topk = recommended[:k]
    hits = sum(1 for item in topk if item in relevant)
    return hits / k

def recall_at_k(recommended, relevant, k):
    """Fraction of relevant items in top-k recommendations."""
    topk = set(recommended[:k])
    hits = sum(1 for item in relevant if item in topk)
    return hits / len(relevant) if relevant else 0

def f1_at_k(recommended, relevant, k):
    p = precision_at_k(recommended, relevant, k)
    r = recall_at_k(recommended, relevant, k)
    return 2*p*r/(p+r) if p+r > 0 else 0

# Example: Alice's true preferences and our ranking
recommended = ['M3','M1','M5','M2','M4','M6']
relevant     = {'M1','M3','M6'}  # items Alice actually liked

print("Ranking-based metrics for Alice:")
for k in [1,2,3,4,5,6]:
    p = precision_at_k(recommended, relevant, k)
    r = recall_at_k(recommended, relevant, k)
    f1 = f1_at_k(recommended, relevant, k)
    print(f"  k={k}: Precision={p:.3f}, Recall={r:.3f}, F1={f1:.3f}")

Ranking-based metrics for Alice:
  k=1: Precision=1.000, Recall=0.333, F1=0.500
  k=2: Precision=1.000, Recall=0.667, F1=0.800
  k=3: Precision=0.667, Recall=0.667, F1=0.667
  k=4: Precision=0.500, Recall=0.667, F1=0.571
  k=5: Precision=0.400, Recall=0.667, F1=0.500
  k=6: Precision=0.500, Recall=1.000, F1=0.667


In [3]:
def dcg(scores):
    """Discounted Cumulative Gain."""
    return sum(s/math.log2(i+2) for i,s in enumerate(scores))

def ndcg_at_k(recommended, relevance_scores, k):
    """Normalized DCG at k. relevance_scores: {item: score}"""
    topk = recommended[:k]
    gains = [relevance_scores.get(item, 0) for item in topk]
    ideal_gains = sorted(relevance_scores.values(), reverse=True)[:k]
    ideal_dcg = dcg(ideal_gains)
    return dcg(gains)/ideal_dcg if ideal_dcg > 0 else 0

# Graded relevance: 5=loved, 3=liked, 1=neutral
relevance = {'M1':5, 'M3':5, 'M6':3, 'M2':1}
print("NDCG@k:")
for k in [1,2,3,4,5,6]:
    ndcg = ndcg_at_k(recommended, relevance, k)
    print(f"  k={k}: NDCG={ndcg:.4f}")

NDCG@k:
  k=1: NDCG=1.0000
  k=2: NDCG=1.0000
  k=3: NDCG=0.8446
  k=4: NDCG=0.8513
  k=5: NDCG=0.8513
  k=6: NDCG=0.9572


## Beyond Accuracy

Accurate != good recommender system. Other dimensions:

| Metric | Meaning | Example |
|--------|---------|---------|
| **Coverage** | Fraction of items ever recommended | Prevents popularity bias |
| **Novelty** | How novel/unexpected are recommendations | Recommend less-popular but relevant |
| **Serendipity** | Unexpectedly relevant recommendations | Diverse recommendations |
| **Diversity** | How different are items within a recommendation list | Avoid redundant lists |

**The accuracy–diversity trade-off:** models optimizing RMSE tend to recommend popular items.
Netflix famously found that users' self-reported wants (what they said they'd watch) differed from what they actually watched — accuracy alone doesn't capture satisfaction.